# Day 34 — scikit-learn intro: API & pipelines
Objectives:
- fit/predict/transform basics.
- Pipeline with preprocessing + model.
- Train/validation split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
X, y = load_diabetes(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X,y, test_size=0.2, random_state=42)
pipe = Pipeline([('sc', StandardScaler()), ('lr', LinearRegression())])
pipe.fit(Xtr,ytr)
pipe.score(Xte,yte)


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — the scikit-learn estimator contract and leakage-safe pipelines

### Mental model

Scikit-learn separates **learning state** from **using learned state**.
`fit` estimates parameters from training data. `transform` applies a
fitted representation, and `predict` applies a fitted predictor.
Attributes ending in an underscore, such as `mean_` or `coef_`, are
commonly learned during fitting.

A `Pipeline` is a single estimator whose fit sequence keeps every
learned preprocessing step inside the training boundary. During
cross-validation, each fold receives a newly fitted pipeline. This
prevents validation data from influencing means, encodings, feature
selection, or other learned state.

### Read the API before running it

- **`Pipeline([('scale', ...), ('model', ...)])`:** names ordered steps so nested parameters and learned attributes remain inspectable.
- **`.fit(X_train, y_train)`:** fits every transformer on training rows, transforms those rows, then fits the final estimator.
- **`.predict(X_new)`:** uses already learned preprocessing and model state; it must not refit on new data.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — prove that the scaler learned only training data

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Train/test membership was decided before any learned transformation.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_train = np.array([[0.0], [2.0], [4.0]])
y_train = np.array([0.0, 2.0, 4.0])
X_test = np.array([[100.0]])
pipe = Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
pipe.fit(X_train, y_train)

learned_mean = pipe.named_steps["scale"].mean_[0]
prediction = pipe.predict(X_test)[0]
print({"training_mean": learned_mean, "prediction": prediction})
assert learned_mean == X_train.mean()

**Expected observation:** The scaler mean is `2.0`; the extreme test row never influenced fitted preprocessing.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — handle an unseen category at prediction time

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** An all-zero encoded unknown has an acceptable, documented meaning for the downstream model.

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train = np.array([["red"], ["blue"], ["red"]])
encoder.fit(train)
transformed = encoder.transform(np.array([["green"], ["red"]]))
print(encoder.categories_, transformed)
assert transformed.shape == (2, 2)
assert transformed[0].sum() == 0

**Expected observation:** The unknown `green` row becomes all zeros instead of crashing or inventing a learned category.

### Debugging and practice ramp

**Common mistake:** Calling `fit_transform` on the entire dataset before splitting or cross-validation.

**Diagnostic:** Inspect every learned underscore attribute and ask which row IDs contributed to it; wrap all learned preprocessing in the pipeline.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define the scikit-learn estimator contract and leakage-safe pipelines in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not report evaluation results until the split happened before fitting and the complete feature pipeline was evaluated.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Swap `LinearRegression` for `Ridge` and compare test scores.

**Verify:** For task `Swap LinearRegression for Ridge and compare test scores`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






2. Inspect the coefficients and discuss how feature scaling changes their
   numeric values and interpretation.

**Verify:** For task `Inspect the coefficients and discuss how feature scaling changes their`, demonstrate the concrete requirement “2. Inspect the coefficients and discuss how feature scaling changes their numeric values and interpretation” with explicit inputs, observable output, and one counterexample.







### Progressive hints

1. Keep the split and scaler fixed; change only the final named step. Start with
   `alpha=1.0`, then record both models rather than declaring a winner from one
   unexplained number.
2. Reach the fitted model through `pipeline.named_steps`. A coefficient from
   standardized inputs represents a one-standard-deviation feature change, but
   correlated features still complicate causal interpretation.

### Additional mastery practice

Make preprocessing and estimation one fitted object. Data boundaries, feature names, and unknown-category behavior are part of the model contract.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Leakage prediction:** Predict how cross-validation scores can change when a scaler is fit on the complete dataset before `cross_val_score`, then explain why the code still runs without warning.
   **Progressive hint:** The globally fitted mean and scale contain information from each validation fold. A Pipeline refits them using only the fold's training rows.

**Verify:** For task `Leakage prediction: Predict how cross-validation scores can change when a scaler is fit on th...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







4. **Mixed-type implementation:** Build a `ColumnTransformer` for numeric imputation/scaling and categorical imputation/one-hot encoding, followed by LogisticRegression. Use a tiny DataFrame containing a missing value.
   **Progressive hint:** Use separate nested pipelines and `handle_unknown='ignore'`; keep column lists explicit so schema drift is visible.

**Verify:** For task `Mixed-type implementation: Build a ColumnTransformer for numeric imputation/scaling and categ...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.







5. **Unknown-category debugging:** Fit on regions `north` and `south`, then predict a row with region `west`. Compare `OneHotEncoder` default behavior with `handle_unknown='ignore'` and explain the resulting representation.
   **Progressive hint:** The default raises on an unseen category. Ignore maps the unknown to all zeros for that feature block, which is operationally safe but lossy.

**Verify:** For task `Unknown-category debugging: Fit on regions north and south, then predict a row with region we...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case.







6. **Inspection and schema contract:** After fitting the mixed-type pipeline, recover transformed feature names, pair them with coefficients, and assert that an inference DataFrame has the required columns in a safe order.
   **Progressive hint:** Use `get_feature_names_out()` from the fitted ColumnTransformer. Select by column name rather than trusting an incoming positional order.

**Verify:** For task `Inspection and schema contract: After fitting the mixed-type pipeline, recover transformed fe...`, report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels; then assert exact names, order, types/nullability or versions and prove one mismatch is rejected rather than silently coerced.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Leakage prediction


# Practice 4 — Mixed-type implementation


# Practice 5 — Unknown-category debugging


# Practice 6 — Inspection and schema contract
